# 04 · Dimensiones restantes y carga de `fact_ventas`
**IDY1102 · EP3 · DataRetail 360**

| Campo | Detalle |
|---|---|
| Responsables | [Mara Cantillana, Cristobal Cardenas, Victor Navarrete]  |
| Se ejecuta | **Al final**, cuando 01, 02 y 03 ya cargaron todos los snapshots |
| Dimensiones simples | `dim_fecha` (ejemplo resuelto), `dim_canal`, `dim_sucursal` (SCD Tipo 1) |
| Hecho | `fact_ventas`, grano: una línea de producto por boleta u orden |
| Fuentes | `ventas_pos.csv` (tiendas) y `ventas_ecommerce.json` (canal digital, anidado) |

> Aquí se une todo: cada línea de venta busca la **versión de la dimensión vigente a la fecha de la venta**
> (`db.asignar_clave_scd2`). Las celdas **✏️ TODO** son las reglas de calidad y las búsquedas de claves.

## 0. Setup

In [18]:
import os, sys
from datetime import date
from pathlib import Path

import pandas as pd

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ))

from src import db, extract, medallion, quality as q, schema
from src.config import get_settings
from src.logger import get_logger

PROCESO = "integracion"
log = get_logger(PROCESO)
cfg = get_settings()
engine = db.get_engine(cfg)
schema.crear_esquema(engine)                  # idempotente: crea solo lo que falta
schema.sembrar_miembros_desconocidos(engine)  # miembro -1 "No informado" en cada dimensión
pd.set_option("display.width", 160)

2026-10-25 21:16:54,438 | INFO     | datarretail360.db | Conectado a sqlserver (DataRetail360_DW)


## A. `dim_fecha` — ejemplo resuelto
Se genera (no viene de una fuente). Sirve de modelo para el patrón *extract → transform → load → validar*.

In [19]:
def construir_dim_fecha(desde: str = "2024-01-01", hasta: str = "2026-12-31") -> pd.DataFrame:
    f = pd.DataFrame({"fecha": pd.date_range(desde, hasta)})
    f["fecha_key"] = f["fecha"].dt.strftime("%Y%m%d").astype(int)      # clave inteligente AAAAMMDD
    f["dia"] = f["fecha"].dt.day
    f["mes"] = f["fecha"].dt.month
    f["nombre_mes"] = f["fecha"].dt.month_name()
    f["trimestre"] = f["fecha"].dt.quarter
    f["anio"] = f["fecha"].dt.year
    f["dia_semana"] = f["fecha"].dt.day_name()
    f["es_fin_de_semana"] = (f["fecha"].dt.dayofweek >= 5).astype(int)
    return f

with db.Ejecucion(engine, "dim_fecha") as run:
    fechas = construir_dim_fecha()
    run.leidas = len(fechas)
    r = db.insertar_si_no_existe(engine, schema.dim_fecha, fechas, ["fecha_key"])   # idempotente
    run.cargadas = r["insertadas"]
r

2026-10-25 21:17:02,804 | INFO     | datarretail360.db | [dim_fecha] INICIO
2026-10-25 21:17:03,223 | INFO     | datarretail360.db | Carga dim_fecha: {'insertadas': 0, 'ya_existian': 1096, 'duplicadas_en_lote': 0}
2026-10-25 21:17:03,227 | INFO     | datarretail360.db | [dim_fecha] FIN OK leidas=1096 cargadas=0 rechazadas=0 (0.4s)


{'insertadas': 0, 'ya_existian': 1096, 'duplicadas_en_lote': 0}

## B. `dim_canal`  ✏️ TODO
Dimensión Tipo 1 con pocos valores. Los canales de las fuentes vienen escritos de varias formas (`Tienda física`, `TIENDA FÍSICA`, `tienda fisica`, `E-commerce`, `ecommerce`, `Web`).

In [20]:
def canal_canonico(serie: pd.Series) -> pd.Series:
    """Lleva cualquier variante de la fuente a su canal canónico."""
    s = q.normalizar_texto(serie, mayusculas=True, sin_tildes=True)
    return s.map(lambda v: "Tienda física" if str(v).startswith("TIENDA") else "E-commerce")

def cargar_dim_canal() -> dict:
    df = pd.DataFrame({"canal": ["Tienda física", "E-commerce"], "tipo_canal": ["Físico", "Digital"]})
    return db.aplicar_scd1(engine, "dim_canal", df, "canal", ["tipo_canal"], "canal_key")

## C. `dim_sucursal`  ✏️ TODO
Lee `sucursales.csv`, resuelve `geografia_key` desde `codigo_comuna` (`dim_geografia` ya cargada) y carga con SCD Tipo 1.

In [21]:
def cargar_dim_sucursal() -> dict:
    s = extract.leer_csv(cfg.data_dir / "sucursales.csv")
    mapa = db.mapa_claves(engine, "dim_geografia", "codigo_comuna", "geografia_key")
    s["geografia_key"] = s["codigo_comuna"].map(mapa).fillna(-1).astype(int)
    return db.aplicar_scd1(engine, "dim_sucursal", s, "id_sucursal",
                           ["nombre_sucursal", "tipo_tienda", "geografia_key"], "sucursal_key")

## D. `fact_ventas`

### D.1 Extract
Une las dos fuentes en un solo esquema. El JSON de e-commerce viene anidado (`items` dentro de cada orden) y se aplana con `leer_json`.

In [22]:
COLUMNAS_COMUNES = ["id_transaccion", "linea", "fecha", "sku", "rut", "id_sucursal", "canal",
                    "cantidad", "monto_total", "estado", "origen_archivo", "fila_origen"]

def extraer_ventas() -> pd.DataFrame:
    """Lee POS (CSV) y e-commerce (JSON) y las deja en el mismo esquema. Guarda el crudo en Bronze."""
    pos = extract.leer_csv(cfg.data_dir / "ventas_pos.csv").rename(columns={"monto": "monto_total"})
    ec = extract.leer_json(cfg.data_dir / "ventas_ecommerce.json", record_path="items",
                           meta=["order_id", "created_at", "channel", "status", ["customer", "rut"]])
    ec = ec.rename(columns={"order_id": "id_transaccion", "created_at": "fecha", "channel": "canal",
                            "qty": "cantidad", "line_total": "monto_total", "customer_rut": "rut"})
    ec["id_sucursal"] = "S00"                                   # el canal digital sale del centro de distribución
    ec["estado"] = ec["status"].map({"PAID": "OK", "RETURNED": "DEVUELTA", "CANCELLED": "ANULADA"})
    # Las líneas se numeran DENTRO de cada aparición de la orden: si una orden viene repetida, sus líneas
    # repiten (id_transaccion, linea) y la regla de líneas duplicadas del Transform las detecta.
    ec["linea"] = (ec.groupby(["id_transaccion", "_registro"]).cumcount() + 1).astype("string")   # Bronze = todo texto
    crudo = pd.concat([pos[COLUMNAS_COMUNES], ec[COLUMNAS_COMUNES]], ignore_index=True)
    medallion.guardar_capa(crudo, "bronze", "ventas")
    return crudo

### D.2 Transform  ✏️ TODO
Reglas de calidad que traen los datos de ejemplo:

| Problema en los datos de ejemplo | Dimensión de calidad | Pista |
|---|---|---|
| Líneas repetidas en el POS (misma `id_transaccion` + `linea`, 2 filas) | Unicidad | `duplicated(["id_transaccion", "linea"])` |
| Una orden de e-commerce aparece dos veces en el JSON (`EC-1002`) | Unicidad | sus líneas repiten `(id_transaccion, linea)`: la misma regla las atrapa (ver cómo numera `extraer_ventas`) |
| Fechas en 4 formatos (y una imposible, `31-02-2025`) | Validez | `q.parsear_fechas` y rechazar las `NaT` |
| Montos negativos con estado OK (2 filas) | Validez | `monto_total < 0` → rechazo |
| Ventas anuladas (POS y e-commerce) | Consistencia con el negocio | decidir y documentar: ¿se rechazan o se cargan marcadas? |
| Devoluciones | Consistencia | columna `devolucion` 0/1 (como en el Diccionario 2.1.5) |
| SKU escritos distinto, y un SKU que no existe en el catálogo | Consistencia | `q.normalizar_sku`; lo desconocido va a la clave `-1` |
| Canal escrito de varias formas | Consistencia | la función de canal de la sección B |
| Sucursal vacía en el POS | Completitud | clave `-1` ("Sucursal no informada") |
| Venta sin RUT (cliente no identificado) | Completitud | `rut_anonimizado` nulo → clave `-1` |
| Monto con IVA incluido | Precisión | `monto_neto = monto_total / 1.19`, `monto_iva = monto_total − monto_neto` (ver `docs/acuerdos_tecnicos.md`) |

In [23]:
# Contrato de salida de asignar_claves(): columnas que espera la carga de fact_ventas (no lo borres)
COLUMNAS_HECHOS = ["id_transaccion", "linea", "fecha_key", "producto_key", "cliente_key", "sucursal_key",
                   "canal_key", "cantidad", "monto_neto", "monto_iva", "monto_total", "devolucion"]

In [24]:
def transformar_ventas(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    from src.config import exigir_rut_salt
    df = df.copy()
    rechazos = []
    for col in ["linea", "cantidad", "monto_total"]:
        df[col] = pd.to_numeric(df[col])

    df, r = q.separar_rechazos(df, df.duplicated(["id_transaccion", "linea"], keep="first"), "linea_duplicada"); rechazos.append(r)
    df["fecha"] = q.parsear_fechas(df["fecha"])
    df, r = q.separar_rechazos(df, df["fecha"].isna(), "fecha_invalida"); rechazos.append(r)
    df, r = q.separar_rechazos(df, df["monto_total"] < 0, "monto_negativo"); rechazos.append(r)
    df, r = q.separar_rechazos(df, df["estado"] == "ANULADA", "venta_anulada"); rechazos.append(r)

    df["sku"] = q.normalizar_sku(df["sku"])
    df["canal"] = canal_canonico(df["canal"])
    df["id_sucursal"] = q.normalizar_texto(df["id_sucursal"]).fillna("-1")
    df["rut_anonimizado"] = q.anonimizar_serie(df["rut"], exigir_rut_salt())
    df["monto_neto"] = (df["monto_total"] / 1.19).round(0)
    df["monto_iva"] = df["monto_total"] - df["monto_neto"]
    df["devolucion"] = (df["estado"] == "DEVUELTA").astype(int)
    return df, q.consolidar_rechazos(*rechazos)

### D.3 Claves sustitutas  ✏️ TODO
Aquí se ve el valor del SCD2: cada venta toma la versión de **producto** y **cliente** vigente **a su fecha**.

In [25]:
def asignar_claves(ventas: pd.DataFrame) -> pd.DataFrame:
    v = ventas.copy()
    v["fecha_key"] = v["fecha"].dt.strftime("%Y%m%d").astype(int)
    v["producto_key"] = db.asignar_clave_scd2(v, db.leer_tabla(engine, "dim_producto"), "sku", "fecha", "producto_key")
    v["cliente_key"] = db.asignar_clave_scd2(v, db.leer_tabla(engine, "dim_cliente"), "rut_anonimizado", "fecha", "cliente_key")
    v["sucursal_key"] = v["id_sucursal"].map(db.mapa_claves(engine, "dim_sucursal", "id_sucursal", "sucursal_key")).fillna(-1).astype(int)
    v["canal_key"] = v["canal"].map(db.mapa_claves(engine, "dim_canal", "canal", "canal_key")).fillna(-1).astype(int)
    return v

### D.4 Load y ejecución
Carga idempotente por `(id_transaccion, linea)`: ejecutar dos veces no duplica.

In [26]:
def cargar_hechos(hechos: pd.DataFrame, rechazos: pd.DataFrame) -> dict:
    assert set(COLUMNAS_HECHOS) <= set(hechos.columns), "asignar_claves() no devolvió las columnas esperadas"
    db.registrar_rechazos(engine, "fact_ventas", rechazos)
    return db.insertar_si_no_existe(engine, schema.fact_ventas, hechos[COLUMNAS_HECHOS], ["id_transaccion", "linea"])

In [27]:
cargar_dim_canal()
cargar_dim_sucursal()

with db.Ejecucion(engine, "fact_ventas") as run:
    crudo = extraer_ventas()
    run.leidas = len(crudo)
    validas, rechazos = transformar_ventas(crudo)
    medallion.guardar_capa(validas, "silver", "ventas")
    hechos = asignar_claves(validas)
    resumen = cargar_hechos(hechos, rechazos)
    run.cargadas = resumen["insertadas"]
    run.rechazadas = len(rechazos)
resumen

2026-10-25 21:17:24,828 | INFO     | datarretail360.db | SCD1 dim_canal: {'nuevos': 0, 'actualizados': 0, 'sin_cambio': 2}


2026-10-25 21:17:24,856 | INFO     | datarretail360.extract | Extraídas 7 filas desde sucursales.csv
2026-10-25 21:17:24,947 | INFO     | datarretail360.db | SCD1 dim_sucursal: {'nuevos': 7, 'actualizados': 0, 'sin_cambio': 0}
2026-10-25 21:17:24,949 | INFO     | datarretail360.db | [fact_ventas] INICIO
2026-10-25 21:17:24,964 | INFO     | datarretail360.extract | Extraídas 85 filas desde ventas_pos.csv
2026-10-25 21:17:24,983 | INFO     | datarretail360.extract | Extraídas 26 filas desde ventas_ecommerce.json
2026-10-25 21:17:25,365 | INFO     | datarretail360.medallion | Capa bronze: 111 filas guardadas en ventas.parquet
2026-10-25 21:17:25,902 | INFO     | datarretail360.medallion | Capa silver: 101 filas guardadas en ventas.parquet
2026-10-25 21:17:26,161 | INFO     | datarretail360.db | Carga auditoria_rechazos: {'insertadas': 10, 'ya_existian': 0, 'duplicadas_en_lote': 0}
2026-10-25 21:17:26,163 | WARNING  | datarretail360.db | [fact_ventas] 10 filas rechazadas (nuevas en auditor

{'insertadas': 101, 'ya_existian': 0, 'duplicadas_en_lote': 0}

## 5. Validación
Conciliación de conteos y de montos: lo que entró = lo que se cargó + lo que se rechazó.

In [28]:
def validar_hechos():
    n_leidas, n_validas, n_rech = len(crudo), len(validas), len(rechazos)
    assert n_leidas == n_validas + n_rech, f"No concilia: {n_leidas} leídas ≠ {n_validas} válidas + {n_rech} rechazadas"
    cargados = db.leer_tabla(engine, "fact_ventas")
    assert cargados.set_index(["id_transaccion", "linea"]).index.is_unique, "hay líneas duplicadas en el DW"
    assert abs(cargados["monto_total"].sum() - hechos["monto_total"].sum()) < 1, "el monto cargado no coincide con el esperado"
    sin_dato = (cargados[["producto_key", "cliente_key", "sucursal_key", "canal_key"]] == -1).any(axis=1).mean()
    print(f"Conciliación OK · {n_leidas} = {n_validas} + {n_rech} · hechos con alguna clave 'No informado': {sin_dato:.1%}")

validar_hechos()

Conciliación OK · 111 = 101 + 10 · hechos con alguna clave 'No informado': 9.9%


In [29]:
# Idempotencia: volver a cargar los mismos hechos no inserta nada
filas_antes = len(db.leer_tabla(engine, "fact_ventas"))
r2 = cargar_hechos(hechos, pd.DataFrame(columns=q.COLUMNAS_RECHAZO))
assert len(db.leer_tabla(engine, "fact_ventas")) == filas_antes and r2["insertadas"] == 0
print("Idempotencia OK:", r2)

2026-10-25 21:17:38,878 | INFO     | datarretail360.db | Carga fact_ventas: {'insertadas': 0, 'ya_existian': 101, 'duplicadas_en_lote': 0}
Idempotencia OK: {'insertadas': 0, 'ya_existian': 101, 'duplicadas_en_lote': 0}


## 6. Evidencia
Una consulta de negocio sobre el esquema estrella demuestra que el modelo responde las preguntas del caso.

In [30]:
consulta = """
SELECT g.region, p.categoria, SUM(f.monto_neto) AS venta_neta, SUM(f.cantidad) AS unidades
FROM fact_ventas f
JOIN dim_sucursal s ON s.sucursal_key = f.sucursal_key
JOIN dim_geografia g ON g.geografia_key = s.geografia_key
JOIN dim_producto p ON p.producto_key = f.producto_key
WHERE f.devolucion = 0
GROUP BY g.region, p.categoria
ORDER BY venta_neta DESC
"""
with engine.connect() as conn:
    display(pd.read_sql(consulta, conn).head(10))

,region,categoria,venta_neta,unidades
0,Metropolitana,Tecnología,26451934.0,42
1,Valparaíso,Tecnología,5082278.0,11
2,Araucanía,Tecnología,4913404.0,8
3,Biobío,Tecnología,3418429.0,9
4,Biobío,Hogar,2899060.0,12
5,Antofagasta,Tecnología,2334437.0,4
6,Antofagasta,Hogar,2226823.0,8
7,Araucanía,Hogar,1705823.0,7
8,Metropolitana,Hogar,1562942.0,10
9,Metropolitana,Accesorios,586344.0,25


In [31]:
# Trazabilidad: ejecuciones y rechazos de TODO el pipeline
ejec = db.leer_tabla(engine, "auditoria_ejecuciones")
display(ejec.tail(10)[["proceso", "inicio", "estado", "filas_leidas", "filas_cargadas", "filas_rechazadas"]])
rech = db.leer_tabla(engine, "auditoria_rechazos")
rech.groupby(["proceso", "motivo_rechazo"]).size().rename("filas").to_frame()

,proceso,inicio,estado,filas_leidas,filas_cargadas,filas_rechazadas
1,dim_producto,2026-10-25 18:54:34.270,ERROR,12,0,0
2,dim_producto,2026-10-25 19:00:29.883,OK,12,10,2
3,dim_producto,2026-10-25 19:14:49.043,OK,11,3,0
4,dim_cliente,2026-10-25 19:57:12.070,OK,30,26,4
5,dim_geografia,2026-10-25 19:58:24.623,ERROR,16,0,0
6,dim_geografia,2026-10-25 20:56:10.040,ERROR,16,0,0
7,dim_geografia,2026-10-25 21:03:27.447,OK,16,15,1
8,dim_fecha,2026-10-25 21:07:04.543,OK,1096,1096,0
9,dim_fecha,2026-10-25 21:17:02.803,OK,1096,0,0
10,fact_ventas,2026-10-25 21:17:24.950,OK,111,101,10


filas
proceso       motivo_rechazo                
dim_cliente   cliente_duplicado            2
              rut_invalido                 2
dim_geografia Codigo comuna duplicado      1
dim_producto  sku_duplicado                1
              sku_vacio                    1
fact_ventas   fecha_invalida               1
              linea_duplicada              4
              monto_negativo               2
              venta_anulada                3

## 7. Decisiones de diseño (completar)
- Ventas anuladas: ¿rechazo o carga marcada? ¿Por qué?
- Devoluciones como columna (`devolucion`) vs. tabla de hechos separada (`Fact_Devoluciones` del EP1): ¿qué ganamos y qué perdemos?
- Claves `-1`: ¿qué porcentaje de hechos queda con alguna dimensión «No informada» y qué significa para el análisis?